# 04 - Build ChromaDB Regulatory Knowledge Base

**ITAI 2376 Final Project - GridGuard-AI**  
**Authors:** DeMarcus Crump & Yoana Cook

Indexes the 28 NERC / ERCOT / FERC regulatory PDFs in `data/regulatory_docs/` into a local **ChromaDB** vector store that the Regulatory Knowledge agent queries at runtime via semantic similarity (RAG).

### Pipeline
1. Walk `data/regulatory_docs/` and extract text from every PDF with `pypdf`.
2. Chunk each document into ~800-token passages with 120-token overlap.
3. Embed each chunk using `sentence-transformers/all-MiniLM-L6-v2` (local, free, 384-dim).
4. Persist the collection to `data/chroma_db/` - this folder is committed to git so the runtime agent does not have to re-embed on every cold-start.

### Deep-learning connection
The embedding model is a **Transformer** (MiniLM) fine-tuned for sentence similarity. Each regulatory clause becomes a dense vector, and retrieval is nearest-neighbour cosine similarity - direct application of the embeddings and attention-weighted representation concepts from Module 05.

Expected runtime in Colab: ~3-5 min (depends on PDF page count).

In [1]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip -q install chromadb==0.4.24 sentence-transformers==2.7.0 pypdf==4.2.0
    REPO_URL = os.environ.get('GRIDGUARD_REPO_URL', 'https://github.com/crumpdemarcus/GridGuard-AI.git')
    if not os.path.isdir('/content/repo'):
        !git clone {REPO_URL} /content/repo
    os.chdir('/content/repo')
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
print('Working directory:', os.getcwd())

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.5/525.5 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.5 MB/s

In [3]:
import glob, re, json, hashlib
from pathlib import Path
from pypdf import PdfReader

# Patch for NumPy 2.0 compatibility with older ChromaDB
import numpy as np
if not hasattr(np, 'float_'):
    np.float_ = np.float64

import chromadb
from chromadb.utils import embedding_functions

PDF_ROOT    = Path('data/regulatory_docs')
CHROMA_DIR  = Path('data/chroma_db')
COLLECTION  = 'gridguard_regulatory_kb'
CHUNK_WORDS = 500         # ~800 tokens
OVERLAP     = 80

CHROMA_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(PDF_ROOT.rglob('*.pdf'))
print(f'Found {len(pdfs)} PDFs under {PDF_ROOT}')
for p in pdfs:
    print(f'  {p.relative_to(PDF_ROOT)}')


Found 28 PDFs under data/regulatory_docs
  ai_governance/NIST_AI_RMF_1.0.pdf
  ai_governance/White_House_AI_EO_14110.pdf
  doe_energy/DOE_Grid_Modernization.pdf
  ercot_protocols/ERCOT_CDR_May2025.pdf
  ercot_protocols/ERCOT_MORA_March2026.pdf
  ercot_protocols/ERCOT_Nodal_Operating_Guide_Feb2026.pdf
  ercot_protocols/ERCOT_Nodal_Protocols_April2026.pdf
  ferc_regulations/FERC_Order_2222_DER_Aggregation.pdf
  ferc_regulations/FERC_Order_841_Energy_Storage.pdf
  historical_incidents/FERC_NERC_Cold_Weather_2021.pdf
  historical_incidents/FERC_NERC_Elliott_2022.pdf
  historical_incidents/FERC_NERC_Uri_2021.pdf
  historical_incidents/Northeast_Blackout_2003.pdf
  nerc_standards/NERC_BAL-001-2.pdf
  nerc_standards/NERC_BAL-002-3.pdf
  nerc_standards/NERC_CIP-014-3.pdf
  nerc_standards/NERC_EOP-004-4.pdf
  nerc_standards/NERC_EOP-011-3.pdf
  nerc_standards/NERC_EOP-012-3.pdf
  nerc_standards/NERC_FAC-011-4.pdf
  nerc_standards/NERC_IRO-006-5.pdf
  nerc_standards/NERC_MOD-031-3.pdf
  nerc_sta

In [4]:
# --- Text extraction + chunking ---
def extract_text(pdf_path: Path) -> str:
    try:
        reader = PdfReader(str(pdf_path))
        pages = [p.extract_text() or '' for p in reader.pages]
        text  = '\n'.join(pages)
        text  = re.sub(r'\s+', ' ', text).strip()
        return text
    except Exception as exc:
        print(f'  [WARN] failed to parse {pdf_path.name}: {exc}')
        return ''

def chunk(text: str, size: int = CHUNK_WORDS, overlap: int = OVERLAP):
    words = text.split()
    if not words:
        return []
    chunks, step = [], size - overlap
    for i in range(0, len(words), step):
        piece = ' '.join(words[i:i+size])
        if len(piece) > 80:
            chunks.append(piece)
        if i + size >= len(words):
            break
    return chunks

corpus = []   # list of (id, text, metadata)
for pdf in pdfs:
    category = pdf.parent.name
    source   = pdf.name
    text     = extract_text(pdf)
    for idx, piece in enumerate(chunk(text)):
        uid = hashlib.md5(f'{source}-{idx}'.encode()).hexdigest()
        corpus.append((uid, piece, {'source': source, 'category': category, 'chunk_index': idx}))

print(f'\nTotal chunks across {len(pdfs)} PDFs: {len(corpus):,}')


Total chunks across 28 PDFs: 4,071


In [5]:
# --- Build / replace the ChromaDB collection ---
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
try:
    client.delete_collection(COLLECTION)
    print(f'Deleted existing collection "{COLLECTION}".')
except Exception:
    pass

embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)
collection = client.create_collection(
    name=COLLECTION,
    embedding_function=embed_fn,
    metadata={'hnsw:space': 'cosine'},
)

# Batched upsert (Chroma recommends <= 5000 per call)
BATCH = 256
for start in range(0, len(corpus), BATCH):
    batch = corpus[start:start+BATCH]
    collection.add(
        ids       = [row[0] for row in batch],
        documents = [row[1] for row in batch],
        metadatas = [row[2] for row in batch],
    )
    print(f'  embedded {min(start+BATCH, len(corpus)):>6,} / {len(corpus):,}')

print(f'\nCollection "{COLLECTION}" now holds {collection.count():,} chunks.')

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


  embedded    256 / 4,071
  embedded    512 / 4,071
  embedded    768 / 4,071
  embedded  1,024 / 4,071
  embedded  1,280 / 4,071
  embedded  1,536 / 4,071
  embedded  1,792 / 4,071
  embedded  2,048 / 4,071
  embedded  2,304 / 4,071
  embedded  2,560 / 4,071
  embedded  2,816 / 4,071
  embedded  3,072 / 4,071
  embedded  3,328 / 4,071
  embedded  3,584 / 4,071
  embedded  3,840 / 4,071
  embedded  4,071 / 4,071

Collection "gridguard_regulatory_kb" now holds 4,071 chunks.


In [6]:
# --- Sanity-check the RAG retrieval ---
PROBE_QUERIES = [
    'What is the reserve margin threshold that triggers EEA Level 2?',
    'NERC BAL-001 balancing authority requirements',
    'Uri 2021 winter storm generator weatherization lessons',
    'ERCOT system operating limit methodology',
]

for q in PROBE_QUERIES:
    res = collection.query(query_texts=[q], n_results=3)
    print(f'\nQ: {q}')
    for doc, meta, dist in zip(res['documents'][0], res['metadatas'][0], res['distances'][0]):
        print(f'  [{dist:.3f}] {meta["source"]:<42} {doc[:120].strip()}...')

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Q: What is the reserve margin threshold that triggers EEA Level 2?
  [0.487] FERC_NERC_Cold_Weather_2021.pdf            1 might be triggered during the time of the forecasted seasonal peak load. This threshold level was intended to be rough...
  [0.487] FERC_NERC_Uri_2021.pdf                     1 might be triggered during the time of the forecasted seasonal peak load. This threshold level was intended to be rough...
  [0.517] ERCOT_Nodal_Operating_Guide_Feb2026.pdf    Percentages for Level 3 Load shedding will be based on the previous year’s TSP peak Loads, as reported to ERCOT, and wil...

Q: NERC BAL-001 balancing authority requirements
  [0.295] NERC_BAL-001-2.pdf                         Standard BAL-001-2 – Real Power Balancing Control Performance Page 1 of 9 A. Introduction 1. Title: Real Power Balancing...
  [0.379] NERC_BAL-002-3.pdf                         BAL-002-3 – Disturbance Control Standard – Contingency Reserve for Recovery from a Balancing Contingency Event Page 1 of.

In [7]:
# --- Write a small manifest so the runtime agent knows what it is loading ---
manifest = {
    'collection'   : COLLECTION,
    'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
    'n_documents'  : len(pdfs),
    'n_chunks'     : collection.count(),
    'chunk_words'  : CHUNK_WORDS,
    'chunk_overlap': OVERLAP,
    'source_categories': sorted({p.parent.name for p in pdfs}),
    'documents'    : [p.name for p in pdfs],
}
with open(CHROMA_DIR / 'manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=2)

print('Chroma store persisted to', CHROMA_DIR)
print('Manifest:', json.dumps(manifest, indent=2))

if IN_COLAB:
    !zip -r -q chroma_db.zip data/chroma_db
    from google.colab import files
    files.download('chroma_db.zip')
    print('Downloaded chroma_db.zip. Unzip it into your repo so data/chroma_db/ is populated, then commit.')

Chroma store persisted to data/chroma_db
Manifest: {
  "collection": "gridguard_regulatory_kb",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "n_documents": 28,
  "n_chunks": 4071,
  "chunk_words": 500,
  "chunk_overlap": 80,
  "source_categories": [
    "ai_governance",
    "doe_energy",
    "ercot_protocols",
    "ferc_regulations",
    "historical_incidents",
    "nerc_standards",
    "texas_regulations"
  ],
  "documents": [
    "NIST_AI_RMF_1.0.pdf",
    "White_House_AI_EO_14110.pdf",
    "DOE_Grid_Modernization.pdf",
    "ERCOT_CDR_May2025.pdf",
    "ERCOT_MORA_March2026.pdf",
    "ERCOT_Nodal_Operating_Guide_Feb2026.pdf",
    "ERCOT_Nodal_Protocols_April2026.pdf",
    "FERC_Order_2222_DER_Aggregation.pdf",
    "FERC_Order_841_Energy_Storage.pdf",
    "FERC_NERC_Cold_Weather_2021.pdf",
    "FERC_NERC_Elliott_2022.pdf",
    "FERC_NERC_Uri_2021.pdf",
    "Northeast_Blackout_2003.pdf",
    "NERC_BAL-001-2.pdf",
    "NERC_BAL-002-3.pdf",
    "NERC_CIP-014-3.pdf",
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded chroma_db.zip. Unzip it into your repo so data/chroma_db/ is populated, then commit.
